# Brique 2.2 — Encapsulation et visibilité

On repart d'un trou volontaire, laissé ouvert dans le notebook précédent, pour construire ce notebook à partir
d'un vrai problème plutôt que d'une définition abstraite.

## 1. Le problème qu'on n'avait pas résolu : rien ne protège les données

Reprends la classe `CompteBancaire` de l'exercice précédent :

In [ ]:
class CompteBancaire:
    def __init__(self, solde_initial):
        self.solde = solde_initial

    def retirer(self, montant):
        self.solde = self.solde - montant

compte = CompteBancaire(100)
compte.retirer(30)
print(compte.solde)  # 70, normal

# Mais rien n'empêche ceci :
compte.solde = -500  # on modifie directement l'attribut, en contournant totalement retirer()
print(compte.solde)  # -500, un solde négatif absurde, sans qu'aucune règle n'ait été violée au sens de Python

Rien dans la classe `CompteBancaire` n'a été "cassé" au sens où Python l'entend : `self.solde` est un attribut
normal, et un attribut normal, par défaut, se lit et s'écrit librement depuis l'extérieur de la classe, sans
passer par aucune méthode. Même si tu ajoutais une vérification à l'intérieur de `retirer()` ("refuse si le
retrait rend le solde négatif"), ça ne protégerait **que** les appels à `retirer()` — pas une écriture directe
comme `compte.solde = -500`.

**Le vrai problème, nommé précisément** : rien ne garantit ce qu'on appelle un **invariant** — une règle qui doit
rester vraie en permanence pour qu'un objet reste cohérent (ici : "le solde ne doit jamais être négatif"). C'est
exactement ce que l'**encapsulation** va nous permettre de garantir.

## 2. Qu'est-ce que l'encapsulation, en langage simple

L'**encapsulation**, c'est le principe qui dit : les données internes d'un objet ne devraient pas être
modifiables librement depuis l'extérieur — seules les méthodes de la classe elle-même devraient avoir le droit de
les changer, parce que ce sont elles qui savent faire respecter les règles (les invariants) de cet objet.

Une analogie simple : pense à un distributeur de billets. Tu ne peux pas ouvrir toi-même le coffre et prendre des
billets directement — tu passes obligatoirement par l'interface prévue (le clavier, l'écran), qui vérifie que tu
as bien le droit de retirer cette somme avant d'agir. Le coffre est caché, protégé ; seule l'interface officielle
y a accès. L'encapsulation fait la même chose avec les attributs d'un objet : elle cache les données brutes, et
n'autorise à agir dessus qu'à travers des méthodes qui vérifient les règles avant d'agir.

## 3. Comment Java gère ça (ce que tu connais déjà) — et pourquoi Python fait un choix différent

En Java, tu as sûrement déjà écrit `private int solde;`. Le mot `private` est une vraie règle, **vérifiée par le
compilateur** : si un autre bout de code en dehors de la classe essaie d'écrire `compte.solde = -500`, ça ne
compile tout simplement pas. C'est une interdiction stricte, imposée par le langage lui-même, avant même
l'exécution.

**Python fait un choix radicalement différent : il n'impose presque rien par la force.** Il n'existe pas de mot
`private` qui bloquerait vraiment l'accès. À la place, Python utilise des **conventions de nommage** — des
règles suivies par tous les développeurs Python, par discipline partagée, mais que le langage lui-même ne fait
quasiment rien pour faire respecter de force.

Ce choix n'est pas un oubli ni une faiblesse : il vient directement de la philosophie qu'on a construite en
Brique 0 (*Explicit is better than implicit*), résumée par une formule connue de la communauté Python : *"we're
all consenting adults here"* ("nous sommes tous des adultes consentants, ici") — l'idée que Python fait confiance
au développeur plutôt que de l'enfermer derrière des murs imposés par le compilateur. La protection reste
possible et utile, mais elle repose sur des signaux clairs, pas sur une interdiction absolue.

## 4. Premier signal : un underscore simple devant le nom, `_solde`

Par convention, **un seul underscore au début d'un nom d'attribut** (`_solde` au lieu de `solde`) signifie :
"ceci est un détail interne de la classe, ne le lis ni ne l'écris directement depuis l'extérieur, même si Python
te laisserait techniquement le faire".

In [ ]:
class CompteBancaire:
    def __init__(self, solde_initial):
        self._solde = solde_initial   # le underscore signale : "détail interne, ne pas toucher directement"

    def retirer(self, montant):
        self._solde = self._solde - montant

compte = CompteBancaire(100)
print(compte._solde)   # ça fonctionne toujours : Python ne bloque RIEN ici, ce n'est qu'une convention
compte._solde = -500   # toujours possible techniquement, mais quiconque lit ce code sait que c'est une mauvaise pratique

Ce simple underscore ne protège donc **rien** techniquement — c'est un panneau "entrée interdite", pas une porte
verrouillée. Il reste très utile en pratique (il communique l'intention clairement à qui lit le code), mais il ne
résout pas encore notre vrai problème : empêcher un solde négatif.

## 5. Un signal plus fort : le double underscore, et le "name mangling"

Avec **deux underscores au début** (`__solde`), Python fait quelque chose de plus concret que pour un seul
underscore : il **renomme automatiquement** l'attribut en coulisses, en ajoutant le nom de la classe devant. On
appelle ce mécanisme le *name mangling* ("brouillage de nom"). Observons-le directement :

In [ ]:
class CompteBancaire:
    def __init__(self, solde_initial):
        self.__solde = solde_initial

compte = CompteBancaire(100)

try:
    print(compte.__solde)   # ceci échoue : l'attribut ne s'appelle plus vraiment "__solde" depuis l'extérieur
except AttributeError as erreur:
    print("Erreur :", erreur)

# Mais regarde ce qu'il contient réellement :
print(vars(compte))   # vars() affiche tous les attributs réels de l'objet, sans se laisser tromper

Tu vois que l'attribut existe bel et bien, mais sous le nom `_CompteBancaire__solde`, pas `__solde`. C'est ça, le
*name mangling* : Python a automatiquement transformé le nom. Et si tu connais ce nom transformé, rien ne
t'empêche toujours d'y accéder directement (`compte._CompteBancaire__solde = -500` fonctionnerait). **Ce n'est
donc toujours pas une vraie protection** — c'est surtout pensé à l'origine pour éviter des conflits de noms
accidentels quand une classe est héritée (sujet du notebook suivant), pas pour sécuriser des données. En
pratique, la convention à un seul underscore (`_solde`) est largement plus utilisée que le double, justement
parce qu'elle est plus simple et tout aussi honnête sur le fait qu'elle ne "verrouille" rien.

## 6. Le vrai outil pour protéger un invariant : `@property`

On arrive enfin à l'outil qui résout vraiment notre problème de départ. L'idée : on veut que `compte.solde`
**continue de s'utiliser comme un attribut normal** (pas comme `compte.get_solde()`, l'habitude verbeuse qu'on a
souvent en Java), mais qu'en coulisses, **lire ou écrire cette valeur passe en réalité par une méthode**, qui
peut vérifier des règles avant d'agir.

On construit ça en trois étapes, sur notre `CompteBancaire`.

**Étape 1 : stocker la vraie valeur dans un attribut interne (`_solde`, avec un seul underscore, par
convention), et exposer une méthode `solde` qui la lit, décorée avec `@property`.**

`@property`, juste au-dessus d'une méthode, est ce qu'on appelle un **décorateur** : il transforme cette méthode
pour qu'elle se comporte, de l'extérieur, exactement comme un attribut — on l'utilisera comme `compte.solde`,
**sans parenthèses**, même si c'est en réalité une méthode qui s'exécute à chaque lecture.

In [ ]:
class CompteBancaire:
    def __init__(self, solde_initial):
        self._solde = solde_initial

    @property
    def solde(self):
        return self._solde

compte = CompteBancaire(100)
print(compte.solde)   # pas compte.solde() : on l'utilise bien comme un attribut, sans parenthèses

Pour l'instant, cette `@property` ne fait rien de plus qu'un attribut normal — elle se contente de renvoyer
`_solde` tel quel. Mais regarde ce qui se passe si on essaie d'écrire dedans :

In [ ]:
try:
    compte.solde = -500
except AttributeError as erreur:
    print("Erreur :", erreur)

Avec seulement `@property`, l'attribut devient automatiquement **en lecture seule** depuis l'extérieur de la
classe — exactement ce qu'on cherchait. Mais pour notre cas, on veut quand même pouvoir changer le solde (via des
dépôts et retraits), seulement avec une vérification. C'est l'étape suivante.

**Étape 2 : ajouter un "setter" (une méthode d'écriture), avec `@solde.setter`, qui vérifie la règle avant
d'accepter la nouvelle valeur.**

In [ ]:
class CompteBancaire:
    def __init__(self, solde_initial):
        self._solde = solde_initial

    @property
    def solde(self):
        return self._solde

    @solde.setter
    def solde(self, nouvelle_valeur):   # même nom que la property : c'est volontaire, c'est ce qui les relie
        if nouvelle_valeur < 0:
            raise ValueError("Le solde ne peut pas être négatif")   # vu en Brique 1 : lever une exception
        self._solde = nouvelle_valeur

    def retirer(self, montant):
        self.solde = self.solde - montant   # passe maintenant par le setter, donc par la vérification

compte = CompteBancaire(100)
compte.solde = 70        # autorisé : 70 n'est pas négatif, ça passe par le setter, qui met à jour _solde
print(compte.solde)

try:
    compte.solde = -500  # refusé cette fois : le setter vérifie AVANT d'accepter
except ValueError as erreur:
    print("Erreur :", erreur)

try:
    compte.retirer(1000)  # retirer() utilise self.solde = ..., donc passe aussi par la vérification
except ValueError as erreur:
    print("Erreur :", erreur)

Maintenant, **il n'existe plus aucun moyen, depuis l'extérieur de la classe, de rendre le solde négatif** — ni en
écrivant directement `compte.solde = ...` (le setter vérifie), ni en passant par `retirer()` (qui utilise le
setter en interne). L'invariant est vraiment protégé, à un seul endroit, impossible à contourner par erreur.

## 7. Une question honnête : faut-il mettre une `@property` sur chaque attribut, par réflexe ?

Si tu viens de Java, tu as peut-être pris l'habitude d'écrire systématiquement un *getter* et un *setter* pour
chaque attribut, même sans aucune règle à vérifier, "par précaution". **En Python, ce réflexe n'est pas
recommandé.** Rappelle-toi le Zen de Python (Brique 0) : *"simple is better than complex"*.

La bonne pratique en Python est différente, et assez rassurante : **commence toujours par un attribut public
normal** (`self.solde = solde_initial`, sans rien de spécial). Transforme-le en `@property` avec un setter
**seulement le jour où tu as vraiment besoin d'une vérification ou d'un calcul** — et comme on l'a vu, le code
qui utilise `compte.solde` n'a même pas besoin de changer quand tu fais cette transformation plus tard, puisque
`compte.solde` s'écrit pareil avant et après. Tu n'as donc rien à perdre à rester simple au départ.

**Récapitulons les trois niveaux de protection vus dans ce notebook, du plus faible au plus solide :**

| Technique | Protège vraiment ? | Quand l'utiliser |
|---|---|---|
| Attribut public normal (`self.solde`) | Non, aucune protection | Par défaut, tant qu'aucune règle à vérifier |
| Un seul underscore (`self._solde`) | Non, juste un signal de convention | Pour signaler "détail interne", sans urgence particulière |
| Double underscore (`self.__solde`) | Non (juste renommé, contournable) | Rarement, surtout pour éviter des conflits de noms avec l'héritage |
| `@property` + `@x.setter` | **Oui**, vraie vérification à chaque écriture | Dès qu'un invariant doit être garanti (comme ici, le solde) |

---
**Fin du notebook 2.2.** Passe à `01_exercices_easy.ipynb`, puis `02_exercices_medium.ipynb`, avant le notebook
suivant (*Composition*).